# Regulatory Compliance & GDPR Analysis

**Purpose**: Demonstrate GDPR compliance, PII handling, and data privacy capabilities

**Regulatory Focus**:
- GDPR (General Data Protection Regulation)
- PCI-DSS (Payment Card Industry Data Security Standard)
- Data Subject Access Requests (DSAR)
- Right to be Forgotten
- Data Minimization

**Business Questions**:
- What PII do we store and where?
- Can we efficiently handle GDPR data requests?
- Are we compliant with data retention policies?
- What is our PII exposure?

**Target Audience**: Chief Data Officer, Legal, Compliance, DPO

In [ ]:
# Import libraries
import sys
sys.path.append('/home/jovyan/work')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

from middleware import get_db, query, DataTransformers

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print("✓ Libraries imported successfully")

## 1. PII Discovery & Classification

In [ ]:
# Identify all tables and columns containing PII
db = get_db()

# PII Classification
pii_inventory = {
    'customers': {
        'Identifiers': ['customer_id', 'email', 'ssn'],
        'Personal Data': ['first_name', 'last_name', 'phone', 'date_of_birth'],
        'Location Data': ['address', 'city', 'state', 'zip_code'],
        'Financial Data': ['card_number', 'cvv', 'card_expiry'],
        'Security Data': ['password']
    },
    'transactions': {
        'Identifiers': ['customer_id', 'customer_email', 'customer_ssn'],
        'Financial Data': ['card_number', 'cvv', 'amount'],
        'Technical Data': ['ip_address']
    },
    'users': {
        'Identifiers': ['username', 'email'],
        'Security Data': ['password']
    }
}

print("═" * 70)
print("PII DATA INVENTORY")
print("═" * 70)

for table, categories in pii_inventory.items():
    print(f"\n{table.upper()} table:")
    for category, fields in categories.items():
        print(f"  {category:20s}: {', '.join(fields)}")

print("\n" + "═" * 70)
print("⚠️  COMPLIANCE ISSUES DETECTED:")
print("   - CVV stored in database (PCI-DSS violation)")
print("   - Full card numbers stored (should be truncated/tokenized)")
print("   - Passwords not properly hashed (using weak MD5)")
print("   - SSN stored in plain text")
print("═" * 70)

## 2. Data Subject Access Request (DSAR)

Simulate a GDPR Article 15 request - individual's right to access their data

In [ ]:
def execute_dsar(customer_id):
    """
    Execute a Data Subject Access Request for a given customer.
    Returns all personal data we hold about them.
    """
    print(f"Executing DSAR for Customer ID: {customer_id}")
    print("=" * 70)
    
    # Get customer data
    customer = query(f"""
        SELECT * FROM customers 
        WHERE customer_id = '{customer_id}'
    """)
    
    if len(customer) == 0:
        print(f"No customer found with ID: {customer_id}")
        return None
    
    # Get transaction history
    transactions = query(f"""
        SELECT id, amount, status, merchant_id, created_date
        FROM transactions 
        WHERE customer_id = '{customer_id}'
        ORDER BY created_date DESC
    """)
    
    # Compile DSAR response
    dsar_response = {
        'request_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'customer_id': customer_id,
        'personal_information': customer.to_dict('records')[0],
        'transaction_history': transactions.to_dict('records'),
        'data_retention': {
            'customer_since': customer['created_date'].iloc[0],
            'total_transactions': len(transactions),
            'last_activity': transactions['created_date'].max() if len(transactions) > 0 else None
        }
    }
    
    # Display summary
    print("\nPERSONAL INFORMATION:")
    print(f"  Name: {customer['first_name'].iloc[0]} {customer['last_name'].iloc[0]}")
    print(f"  Email: {customer['email'].iloc[0]}")
    print(f"  Customer Since: {customer['created_date'].iloc[0]}")
    
    print("\nTRANSACTION HISTORY:")
    print(f"  Total Transactions: {len(transactions)}")
    if len(transactions) > 0:
        print(f"  Total Spent: ${transactions['amount'].sum():,.2f}")
        print(f"  Last Transaction: {transactions['created_date'].max()}")
    
    print("\nDATA PROCESSING PURPOSES:")
    print("  - Payment processing and transaction execution")
    print("  - Fraud prevention and security")
    print("  - Customer service and support")
    print("  - Legal compliance and record keeping")
    
    print("\nYOUR RIGHTS:")
    print("  - Right to access (this report)")
    print("  - Right to rectification (update your data)")
    print("  - Right to erasure ('right to be forgotten')")
    print("  - Right to restrict processing")
    print("  - Right to data portability")
    print("  - Right to object")
    
    print("\n" + "=" * 70)
    
    return dsar_response

# Example DSAR execution
# Get a sample customer ID
sample_customer = query("SELECT customer_id FROM customers LIMIT 1")
if len(sample_customer) > 0:
    dsar_result = execute_dsar(sample_customer['customer_id'].iloc[0])

## 3. Right to be Forgotten (RTBF)

Demonstrate GDPR Article 17 - Right to Erasure

In [ ]:
def right_to_be_forgotten_analysis(customer_id, execute=False):
    """
    Analyze what would be deleted for a 'Right to be Forgotten' request.
    If execute=True, performs the actual deletion (use with caution!).
    """
    print(f"Right to be Forgotten Analysis for: {customer_id}")
    print("=" * 70)
    
    # Check legal holds or retention requirements
    transactions = query(f"""
        SELECT created_date, amount, status 
        FROM transactions 
        WHERE customer_id = '{customer_id}'
    """)
    
    # Calculate if there are recent transactions (within legal retention period)
    if len(transactions) > 0:
        most_recent = pd.to_datetime(transactions['created_date']).max()
        days_since = (datetime.now() - most_recent).days
        
        # Example: 7-year retention for financial records
        retention_period_days = 7 * 365
        can_delete = days_since >= retention_period_days
    else:
        can_delete = True
        days_since = None
    
    print("\nDELETION SCOPE:")
    print(f"  Customer record: 1 row")
    print(f"  Transaction records: {len(transactions)} rows")
    
    print("\nLEGAL ASSESSMENT:")
    if can_delete:
        print("  ✓ Eligible for deletion")
        print(f"  - All transactions older than {retention_period_days/365:.0f} years")
    else:
        print("  ✗ CANNOT delete - retention requirements apply")
        print(f"  - Most recent transaction: {days_since} days ago")
        print(f"  - Must retain for: {retention_period_days - days_since} more days")
        print(f"  - Legal basis: Financial record retention (7 years)")
    
    print("\nALTERNATIVE ACTIONS:")
    if not can_delete:
        print("  Instead of deletion, we can:")
        print("  1. Pseudonymize the data (replace PII with anonymous ID)")
        print("  2. Mark account for deletion after retention period")
        print("  3. Restrict processing to legal compliance only")
    
    if execute and can_delete:
        print("\n⚠️  EXECUTING DELETION...")
        # In production, this would execute DELETE statements
        # For demo, we just show what would happen
        print("  [DEMO MODE - No actual deletion performed]")
        print("  Would execute:")
        print(f"    DELETE FROM transactions WHERE customer_id = '{customer_id}';")
        print(f"    DELETE FROM customers WHERE customer_id = '{customer_id}';")
    
    print("\n" + "=" * 70)
    
    return {
        'can_delete': can_delete,
        'transaction_count': len(transactions),
        'days_since_last_activity': days_since
    }

# Example analysis (without actual deletion)
if len(sample_customer) > 0:
    rtbf_result = right_to_be_forgotten_analysis(sample_customer['customer_id'].iloc[0], execute=False)

## 4. Data Retention & Lifecycle Analysis

In [ ]:
# Analyze data age for retention compliance
customers = query("SELECT customer_id, created_date FROM customers")
transactions = query("SELECT customer_id, created_date FROM transactions")

customers['created_date'] = pd.to_datetime(customers['created_date'])
transactions['created_date'] = pd.to_datetime(transactions['created_date'])

customers['age_days'] = (datetime.now() - customers['created_date']).dt.days
transactions['age_days'] = (datetime.now() - transactions['created_date']).dt.days

# Retention policy thresholds
retention_policies = {
    'Active Data': 365,  # 1 year
    'Warm Storage': 1095,  # 3 years
    'Cold Archive': 2555,  # 7 years
    'Eligible for Deletion': 2555  # After 7 years
}

# Classify data by retention tier
def classify_retention(age_days):
    if age_days <= 365:
        return 'Active'
    elif age_days <= 1095:
        return 'Warm'
    elif age_days <= 2555:
        return 'Cold'
    else:
        return 'Delete'

customers['retention_tier'] = customers['age_days'].apply(classify_retention)
transactions['retention_tier'] = transactions['age_days'].apply(classify_retention)

# Summary
print("DATA RETENTION ANALYSIS")
print("=" * 70)
print("\nCUSTOMER DATA:")
print(customers['retention_tier'].value_counts())

print("\nTRANSACTION DATA:")
print(transactions['retention_tier'].value_counts())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Customer data age
customer_retention = customers['retention_tier'].value_counts()
axes[0].pie(customer_retention.values, labels=customer_retention.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Customer Data Retention Tiers')

# Transaction data age
transaction_retention = transactions['retention_tier'].value_counts()
axes[1].pie(transaction_retention.values, labels=transaction_retention.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Transaction Data Retention Tiers')

plt.tight_layout()
plt.show()

# Recommendations
delete_eligible = customers[customers['retention_tier'] == 'Delete']
cold_storage = customers[customers['retention_tier'] == 'Cold']

print("\nRECOMMENDATIONS:")
print(f"  - Move {len(cold_storage):,} customer records to cold storage")
print(f"  - Review {len(delete_eligible):,} customer records for deletion")
print(f"  - Archive old transactions to reduce query load")

## 5. Data Minimization Assessment

In [ ]:
# GDPR Principle: Collect only what's necessary
print("DATA MINIMIZATION ASSESSMENT")
print("=" * 70)

# Check for unused or unnecessary data
customers_full = query("SELECT * FROM customers LIMIT 1000")

# Analyze column usage
null_percentage = (customers_full.isnull().sum() / len(customers_full) * 100).sort_values(ascending=False)

print("\nFIELDS WITH HIGH NULL RATES (Potentially unnecessary):")
high_null_fields = null_percentage[null_percentage > 50]
for field, pct in high_null_fields.items():
    print(f"  {field:25s}: {pct:>5.1f}% null")

print("\nCOMPLIANCE VIOLATIONS DETECTED:")
print("  ⚠️  CVV storage: Should NEVER be stored (PCI-DSS 3.2)")
print("  ⚠️  Full card numbers: Should be tokenized or truncated")
print("  ⚠️  SSN: Assess if actually needed for business purpose")
print("  ⚠️  Debug data table: Should not exist in production")

print("\nRECOMMENDATIONS:")
print("  1. Remove CVV field immediately")
print("  2. Implement card tokenization")
print("  3. Review SSN necessity (consider alternatives)")
print("  4. Remove fields with >80% null values")
print("  5. Delete debug_data table from production")
print("=" * 70)

## 6. Data Processing Activities Record (GDPR Article 30)

In [ ]:
# Document processing activities as required by GDPR
processing_activities = [
    {
        'Activity': 'Payment Processing',
        'Purpose': 'Execute customer payment transactions',
        'Legal Basis': 'Contract performance',
        'Data Categories': 'Identity, Financial, Transaction',
        'Recipients': 'Payment processors, banks',
        'Retention': '7 years (legal requirement)',
        'Security': 'Encryption, access controls'
    },
    {
        'Activity': 'Fraud Prevention',
        'Purpose': 'Detect and prevent fraudulent transactions',
        'Legal Basis': 'Legitimate interest',
        'Data Categories': 'Transaction patterns, risk scores',
        'Recipients': 'Fraud detection services',
        'Retention': '3 years',
        'Security': 'Encrypted storage, audit logs'
    },
    {
        'Activity': 'Customer Service',
        'Purpose': 'Provide support and handle inquiries',
        'Legal Basis': 'Contract performance',
        'Data Categories': 'Identity, contact info, transaction history',
        'Recipients': 'Support staff',
        'Retention': '2 years',
        'Security': 'Role-based access control'
    },
    {
        'Activity': 'Analytics & Reporting',
        'Purpose': 'Business intelligence and improvement',
        'Legal Basis': 'Legitimate interest',
        'Data Categories': 'Aggregated transaction data',
        'Recipients': 'Internal analytics team',
        'Retention': 'Anonymized - indefinite',
        'Security': 'Pseudonymization, aggregation'
    }
]

processing_df = pd.DataFrame(processing_activities)

print("RECORD OF PROCESSING ACTIVITIES (ROPA)")
print("=" * 70)
print(processing_df.to_string(index=False))
print("=" * 70)

## 7. Privacy Impact Assessment (PIA)

In [ ]:
# Simple PIA scoring
pia_factors = {
    'Data Sensitivity': {
        'Score': 9,
        'Rating': 'High',
        'Reason': 'Financial data, SSN, card numbers'
    },
    'Data Volume': {
        'Score': 7,
        'Rating': 'Medium-High',
        'Reason': f'{len(customers):,} customers, {len(transactions):,} transactions'
    },
    'Vulnerable Individuals': {
        'Score': 5,
        'Rating': 'Medium',
        'Reason': 'General consumer population'
    },
    'Technology Risks': {
        'Score': 8,
        'Rating': 'High',
        'Reason': 'Legacy system, known vulnerabilities'
    },
    'Third-Party Sharing': {
        'Score': 6,
        'Rating': 'Medium',
        'Reason': 'Payment processors, fraud services'
    }
}

print("PRIVACY IMPACT ASSESSMENT")
print("=" * 70)
print("\nRISK FACTORS (Score: 1-10, 10 = highest risk):\n")

total_score = 0
for factor, details in pia_factors.items():
    print(f"{factor:25s}: {details['Score']}/10 ({details['Rating']})")
    print(f"{'':25s}  → {details['Reason']}")
    total_score += details['Score']

avg_score = total_score / len(pia_factors)
print(f"\nOVERALL RISK SCORE: {avg_score:.1f}/10")

if avg_score >= 7:
    risk_level = 'HIGH'
elif avg_score >= 5:
    risk_level = 'MEDIUM'
else:
    risk_level = 'LOW'

print(f"RISK LEVEL: {risk_level}")

if risk_level == 'HIGH':
    print("\n⚠️  HIGH RISK - Data Protection Impact Assessment (DPIA) REQUIRED")
    print("   Must consult with Data Protection Officer")

print("\nMITIGATION MEASURES NEEDED:")
print("  1. Remove CVV storage immediately")
print("  2. Implement tokenization for card data")
print("  3. Enhance encryption (upgrade from DES to AES-256)")
print("  4. Implement data masking in non-production environments")
print("  5. Regular security audits and penetration testing")
print("  6. Staff privacy training")
print("  7. Incident response plan for data breaches")
print("=" * 70)

## 8. Summary & Action Plan

### Key Findings:

**Compliance Status:**
- ❌ **Critical**: PCI-DSS violations (CVV storage)
- ❌ **High**: Weak encryption, plain text sensitive data
- ✓ **Good**: DSAR and RTBF capabilities functional
- ⚠️ **Medium**: Data retention policies need enforcement

**GDPR Readiness:**
- Legal basis documented for processing activities
- PII inventory complete
- Subject rights requests can be fulfilled
- Privacy impact indicates high-risk processing

**Immediate Risks:**
1. Storing CVV in violation of PCI-DSS
2. Full card numbers stored (should be tokenized)
3. Weak cryptography (DES, MD5)
4. No data masking in development environments

### Action Plan:

**Week 1 (Critical):**
- Stop storing CVV immediately
- Purge existing CVV data
- Notify leadership of PCI-DSS violations

**Month 1 (High Priority):**
- Implement card tokenization
- Upgrade cryptography (AES-256)
- Create production data masking scripts
- Conduct DPIA with DPO

**Quarter 1 (Important):**
- Automate data retention enforcement
- Implement automated DSAR workflow
- Privacy training for all staff
- External security audit

**Ongoing:**
- Monthly compliance reviews
- Quarterly privacy assessments
- Annual penetration testing
- Regular policy updates